# D222 - HDFS NameNode Architecture and High Availability

A detailed guide to NameNode metadata, checkpointing, SecondaryNameNode, HA NameNodes, Quorum Journal Manager (QJM), JournalNodes, ZooKeeper, ZKFC, fencing, and failover. The examples assume Hadoop 3.3.6, while the architecture applies broadly to modern Hadoop 3.x clusters.

> This notebook explains both the single-NameNode teaching setup and a production HA design. Do not enable HA by copying isolated properties: hostnames, storage, security, fencing, and operational testing must be designed together.


## 1. The NameNode's job

HDFS separates **metadata control** from **block data transfer**. The NameNode is the metadata authority; DataNodes store file bytes. A client asks the NameNode to resolve paths and locate or allocate blocks, then streams bytes directly to/from DataNodes. The NameNode is therefore not in the normal data path.

The NameNode manages:

- the filesystem tree: files, directories, symlinks, owners, groups, permissions and ACLs
- quotas, storage policies, erasure-coding policies, snapshots and encryption-zone metadata
- file-to-block IDs and desired replication
- the current locations of block replicas, learned from DataNode block reports
- leases for files being written and recovery of incomplete writes
- DataNode liveness, replication, invalidation and movement commands

It does **not** store ordinary HDFS file contents. Losing NameNode metadata can make intact DataNode blocks unusable because paths and block relationships are lost.


## 2. Single-NameNode architecture and checkpointing

![NameNode, DataNodes, fsimage, edits, and SecondaryNameNode checkpoint flow](images/hdfs-namenode-checkpoint.svg)

The authoritative namespace is held in NameNode memory for fast lookup. Durability comes from two on-disk structures:

| Structure | Meaning |
|---|---|
| `fsimage` | point-in-time serialized namespace checkpoint |
| `edits` | ordered transactions since the checkpoint |
| in-memory namespace | loaded `fsimage` plus replayed edits and subsequent changes |

For a namespace mutation, the NameNode records the edit durably before reporting success. At startup it loads the latest valid image and replays edits. A very long edit log makes restart slower, so periodic checkpoints bound replay work. Block locations are largely rebuilt from DataNode reports rather than stored as a complete durable location map.


## 3. SecondaryNameNode: what it is—and is not

The SecondaryNameNode is a **checkpoint helper**, not a hot standby and not an automatic replacement for a failed NameNode. Its typical cycle is:

1. Ask the NameNode to roll/finalize the current edit log.
2. Download the latest `fsimage` and relevant edits.
3. Load and merge them into a new checkpoint image.
4. Upload the checkpoint back to the NameNode.

This reduces checkpoint workload on the NameNode and limits edit-log replay time. Its checkpoint may help manual recovery, but it might lag and does not continuously serve the namespace. Calling it a backup NameNode is dangerous.

Related roles seen in older/configurable deployments:

- **CheckpointNode** performs checkpoint work similar to the SecondaryNameNode.
- **BackupNode** maintains a more current in-memory namespace by receiving edits, but is still not the same architecture as Hadoop HA with an Active and Standby pair.
- In an HA deployment, the **Standby NameNode performs checkpointing**, so a SecondaryNameNode is normally not used for that nameservice.


## 4. Why high availability is different

A single NameNode is a service availability bottleneck even when its metadata directories are replicated across local disks. HDFS HA runs two NameNodes for one logical **nameservice**:

- **Active:** serves client namespace operations and is the only NameNode allowed to write new edits.
- **Standby:** continuously consumes edits, receives DataNode reports, maintains a hot namespace, makes checkpoints, and can be promoted.

HA is not the same as federation. **HA** removes a NameNode availability single point of failure for one namespace. **Federation** scales/isolate namespaces by running multiple independent nameservices/block pools. Each federated nameservice can itself be an HA pair.


## 5. Production HA with QJM

![HDFS HA with Active and Standby NameNodes, ZKFC, ZooKeeper, JournalNodes and DataNodes](images/hdfs-ha-qjm.svg)

### Component roles

| Component | Role | Does not do |
|---|---|---|
| Active NameNode | serves clients; writes namespace edits | store file block bytes |
| Standby NameNode | tails edits, maintains hot namespace, checkpoints, awaits promotion | accept ordinary writes while Standby |
| JournalNode | persist a copy of the shared edit log | store HDFS user blocks or elect the Active |
| QJM | requires a JournalNode majority for edit-log progress | replace ZooKeeper client coordination |
| ZooKeeper ensemble | maintain coordination state and an ephemeral active lock | store `fsimage`, edit logs, or file blocks |
| ZKFC | monitor its local NameNode and coordinate automatic failover | replicate namespace metadata |
| DataNode | store blocks; report to both NameNodes in the nameservice | decide which NameNode is Active |
| client failover provider | resolve logical nameservice and retry against the Active | elect or fence a NameNode |


## 6. Quorum Journal Manager and JournalNodes

The Active writes each edit to the JournalNode ensemble. With three JournalNodes it needs acknowledgements from a majority—normally 2 of 3—before the edit is committed. The Standby tails committed edits and applies them in order.

Important consequences:

- An odd number, commonly **3 or 5**, provides useful failure tolerance without unnecessary voting ambiguity.
- Three JournalNodes tolerate one unavailable JournalNode; five tolerate two.
- JournalNodes should be on independent failure domains with durable, low-latency disks.
- A JournalNode majority preserves one ordered edit stream using writer epochs, preventing a stale NameNode from successfully continuing to journal.
- Losing quorum stops namespace writes even if the current Active process is alive; reads may behave differently depending on state and configuration.
- The Standby must apply all required committed edits before safe promotion.

QJM protects and shares the **edit log**. Each NameNode still keeps local metadata storage, including images and local state. Checkpoints created by the Standby are made available so both sides can recover efficiently.


## 7. ZooKeeper and ZKFC

Each NameNode host normally runs a **ZooKeeper Failover Controller (ZKFC)**. ZKFC has three conceptual duties:

1. **Health monitoring:** probe the local NameNode and classify it healthy, unhealthy, or unavailable.
2. **ZooKeeper session management:** compete for an ephemeral lock representing the Active role. If its process/session disappears, the lock can be released.
3. **Failover control:** demote/fence the old Active as required, then transition the healthy peer to Active.

ZooKeeper should be a separate odd-sized ensemble, commonly three or five members. ZooKeeper decides coordination ownership; it does not carry HDFS namespace transactions. JournalNodes and ZooKeeper solve different problems and may be colocated only when the failure/capacity design justifies it.


## 8. Automatic failover sequence and split-brain protection

![Automatic HDFS HA failover sequence](images/hdfs-ha-failover.svg)

A typical automatic failover is:

1. ZKFC detects that the Active NameNode is unhealthy or its ZooKeeper session is lost.
2. The peer ZKFC obtains the ZooKeeper active lock.
3. The old Active is fenced if it might still run.
4. The Standby catches up through the JournalNodes.
5. The Standby transitions to Active; configured clients discover/retry against it.

The major risk is **split brain**: two NameNodes both behaving as Active. Protections overlap deliberately:

- ZooKeeper grants the coordination lock to one ZKFC.
- QJM writer epochs reject stale writers at the JournalNodes.
- fencing prevents an old Active from accessing shared resources or serving unsafe work.

Fencing methods may include SSH-based process termination, power/network/storage fencing, or organization-specific scripts. A production fencing command must be idempotent, tightly secured, tested, and able to prove the old writer cannot interfere. Blindly using `sshfence` without reliable SSH privileges and network reachability creates false confidence.


## 9. Normal request paths

### Read

1. Client connects through the logical nameservice URI.
2. Active returns block locations and tokens/authorization information.
3. Client reads directly from a suitable DataNode replica.
4. On a DataNode failure, the client tries another replica.

### Write

1. Active validates path/permissions, grants a lease, selects a DataNode pipeline and journals namespace edits to a JournalNode quorum.
2. Client streams packets through the DataNode pipeline; acknowledgements flow back through it.
3. Active records block/allocation state and handles pipeline recovery when needed.
4. Standby tails committed namespace edits; DataNodes report blocks to both NameNodes.

### Checkpoint in HA

The Standby periodically saves its applied namespace as a new image and coordinates checkpoint availability. This is why the separate SecondaryNameNode is normally unnecessary in HA.


## 10. Representative HA configuration anatomy

The names below illustrate relationships; replace hosts, ports and fencing with the actual design.

```xml
<!-- core-site.xml: clients use a logical nameservice, not one host -->
<property>
  <name>fs.defaultFS</name>
  <value>hdfs://training-ha</value>
</property>

<!-- hdfs-site.xml -->
<property>
  <name>dfs.nameservices</name>
  <value>training-ha</value>
</property>
<property>
  <name>dfs.ha.namenodes.training-ha</name>
  <value>nn1,nn2</value>
</property>
<property>
  <name>dfs.namenode.rpc-address.training-ha.nn1</name>
  <value>nn1.example:8020</value>
</property>
<property>
  <name>dfs.namenode.rpc-address.training-ha.nn2</name>
  <value>nn2.example:8020</value>
</property>
<property>
  <name>dfs.namenode.shared.edits.dir</name>
  <value>qjournal://jn1.example:8485;jn2.example:8485;jn3.example:8485/training-ha</value>
</property>
<property>
  <name>dfs.client.failover.proxy.provider.training-ha</name>
  <value>org.apache.hadoop.hdfs.server.namenode.ha.ConfiguredFailoverProxyProvider</value>
</property>
<property>
  <name>dfs.ha.automatic-failover.enabled</name>
  <value>true</value>
</property>
<property>
  <name>ha.zookeeper.quorum</name>
  <value>zk1.example:2181,zk2.example:2181,zk3.example:2181</value>
</property>
<property>
  <name>dfs.ha.fencing.methods</name>
  <value>sshfence</value>
</property>
```

RPC/service/web addresses for both NameNodes and fencing-specific settings are also normally configured. Never copy placeholder fencing into production.


## 11. Bootstrap and lifecycle—conceptual order

A new QJM-based HA installation commonly follows this controlled sequence:

1. Configure identical nameservice/HA properties on all relevant hosts and clients.
2. Start the JournalNodes and verify a quorum.
3. Format one initial NameNode **only for a genuinely new namespace**.
4. Initialize shared edits.
5. Bootstrap the Standby from the initialized NameNode.
6. Start both NameNodes, DataNodes, ZooKeeper and ZKFC processes.
7. Initialize ZooKeeper HA state once, then test status and failover.

Typical administrative commands include:

```bash
hdfs --daemon start journalnode
hdfs namenode -initializeSharedEdits
hdfs namenode -bootstrapStandby
hdfs zkfc -formatZK
hdfs --daemon start namenode
hdfs --daemon start zkfc
hdfs haadmin -getServiceState nn1
hdfs haadmin -getServiceState nn2
hdfs haadmin -checkHealth nn1
```

> Formatting commands can destroy or replace expected state when used incorrectly. They are shown for architecture learning, not as a copy/paste migration runbook. Existing clusters require a documented, backed-up transition procedure.


## 12. Planned failover, failover testing, and observation

A graceful failover is preferable for maintenance because the system can coordinate transitions:

```bash
hdfs haadmin -failover nn1 nn2
hdfs haadmin -getAllServiceState
```

Forced failover options are emergency tools and may bypass safety checks; use them only with a clear fencing and recovery plan. Test at least:

- planned Active-to-Standby transition
- Active process failure
- Active host/network failure
- one JournalNode failure and quorum restoration
- one ZooKeeper member failure
- client retry behavior during failover
- Standby lag/checkpoint health
- fencing failure, because the unhappy path matters most

Monitor NameNode state, RPC latency, edit transactions, JournalNode quorum/latency, last checkpoint age, Standby transaction lag, ZooKeeper session health, ZKFC decisions, DataNode reports, capacity and under-replicated/missing blocks. Preserve logs from both NameNodes, both ZKFCs, JournalNodes and ZooKeeper around every failover event.


In [ ]:
%%bash
echo '=== Local HDFS processes (single-node lab may not have HA daemons) ==='
jps
echo '=== Configured nameservices ==='
hdfs getconf -confKey dfs.nameservices 2>/dev/null || true
echo '=== Automatic failover ==='
hdfs getconf -confKey dfs.ha.automatic-failover.enabled 2>/dev/null || true
echo '=== Shared edits ==='
hdfs getconf -confKey dfs.namenode.shared.edits.dir 2>/dev/null || true
echo '=== ZooKeeper quorum ==='
hdfs getconf -confKey ha.zookeeper.quorum 2>/dev/null || true
echo '=== NameNode metadata directories ==='
hdfs getconf -confKey dfs.namenode.name.dir 2>/dev/null || true


## 13. Failure scenarios and expected behavior

| Failure | Expected effect | Key requirement |
|---|---|---|
| Active NameNode process/host | peer can become Active | healthy Standby, ZKFC/ZK quorum, fencing |
| Standby NameNode | Active continues; HA redundancy is lost | repair before another failure |
| one of three JournalNodes | writes continue with 2/3 | monitor and restore promptly |
| two of three JournalNodes | no edit quorum; writes cannot safely progress | restore majority without inventing history |
| one of three ZooKeeper nodes | ensemble normally continues with 2/3 | restore coordination redundancy |
| ZooKeeper quorum lost | automatic coordination unavailable; current service may continue, but automatic failover is unsafe/unavailable | restore quorum; avoid manual split brain |
| DataNode | replicas/reads fail over; re-replication scheduled | enough live nodes/capacity and replicas |
| client points to physical NN host | poor/absent transparent failover | logical nameservice + failover provider |
| stale old Active returns | fencing/QJM must prevent writes | tested fencing and epoch enforcement |


## 14. Common misconceptions

- **SecondaryNameNode means standby:** false; it is primarily a checkpointing role.
- **ZooKeeper stores NameNode metadata:** false; ZooKeeper coordinates failover state.
- **JournalNodes store HDFS blocks:** false; they store shared namespace edit logs.
- **RAID/local duplicate name dirs provide HA:** they improve storage durability, not service failover.
- **Two JournalNodes are enough:** there is no useful one-node failure tolerance while retaining majority; three is the common minimum.
- **Two ZooKeeper nodes form robust HA:** losing either loses majority; an odd ensemble is preferred.
- **Standby is idle:** false; it tails edits, processes DataNode reports and checkpoints.
- **HA eliminates backups:** false; HA rapidly reproduces mistakes/corruption too. Keep tested metadata backups, snapshots where appropriate, and disaster-recovery procedures.
- **HA equals federation:** false; availability and namespace scaling are separate dimensions.


## 15. Design and review checklist

- Clients use a logical nameservice URI and the HA failover proxy provider.
- NameNodes are on independent hosts/failure domains with sized heap and durable metadata disks.
- At least three independent JournalNodes provide low-latency quorum storage.
- ZooKeeper is an odd-sized, monitored ensemble with secured access.
- Both ZKFCs run and automatic failover state is initialized.
- Fencing is secure, deterministic, idempotent and tested during realistic partition scenarios.
- DataNodes are configured to report to both NameNodes for the nameservice.
- Standby lag and checkpoint age have alerts.
- Operator procedures distinguish planned failover, forced failover and disaster recovery.
- Metadata backups and restore drills exist outside the HA failure domain.
- Upgrades and configuration changes preserve compatible settings across both NameNodes.
- Regular exercises prove clients recover, quorum loss is understood, and split brain is prevented.


## References

- [Apache Hadoop HDFS Architecture](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/HdfsDesign.html)
- [Apache Hadoop HDFS High Availability with QJM](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/HDFSHighAvailabilityWithQJM.html)
- [Apache Hadoop HDFS Federation](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/Federation.html)
- [Apache Hadoop HDFS Commands Guide](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/HDFSCommands.html)
